In [31]:
import pandas as pd
df=pd.read_csv("SAML-D.csv")

In [32]:
df.shape

(1670771, 12)

In [33]:
"""
AML Dataset Diagnostic EDA - Run this first to understand your data
"""
import pandas as pd
import numpy as np
from scipy.stats import skew
import matplotlib.pyplot as plt
import seaborn as sns

def run_diagnostic_eda(csv_path):
    """
    Run comprehensive diagnostic EDA on AML dataset
    """
    print("🔍 Loading and analyzing AML dataset...")

    # Load data
    df = pd.read_csv(csv_path)
    print(f"Dataset shape: {df.shape}")

    # Basic info
    print("\n📊 BASIC DATASET INFO:")
    print(f"Total transactions: {len(df):,}")
    print(f"Columns: {list(df.columns)}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

    # Target distribution
    print(f"\n🎯 TARGET DISTRIBUTION:")
    target_counts = df['Is_laundering'].value_counts()
    print(f"Normal transactions: {target_counts[0]:,}")
    print(f"Suspicious transactions: {target_counts[1]:,}")
    print(f"Imbalance ratio: 1:{target_counts[0]//target_counts[1]}")

    # Missing values
    print(f"\n❌ MISSING VALUES:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values found")

    # Data types
    print(f"\n📋 DATA TYPES:")
    print(df.dtypes)

    # Amount analysis (most critical)
    print(f"\n💰 AMOUNT ANALYSIS:")
    print(f"Min amount: ${df['Amount'].min():,.2f}")
    print(f"Max amount: ${df['Amount'].max():,.2f}")
    print(f"Mean amount: ${df['Amount'].mean():,.2f}")
    print(f"Median amount: ${df['Amount'].median():,.2f}")
    print(f"Amount skewness: {skew(df['Amount']):.2f}")

    # Large transactions
    large_amounts = df[df['Amount'] > 10000]
    print(f"Transactions > $10k: {len(large_amounts):,} ({len(large_amounts)/len(df)*100:.1f}%)")

    # Just-under-threshold (potential structuring)
    just_under = df[(df['Amount'] >= 9000) & (df['Amount'] < 10000)]
    print(f"Transactions $9k-$10k: {len(just_under):,} ({len(just_under)/len(df)*100:.1f}%)")

    # Categorical features analysis
    print(f"\n🏷️ CATEGORICAL FEATURES:")
    categorical_cols = ['Payment_type', 'Sender_bank_location', 'Receiver_bank_location',
                       'Payment_currency', 'Received_currency']

    for col in categorical_cols:
        if col in df.columns:
            unique_count = df[col].nunique()
            print(f"{col}: {unique_count} unique values")
            if unique_count <= 20:  # Show values if not too many
                print(f"  Values: {list(df[col].unique())}")

    # Payment type vs laundering
    print(f"\n💳 PAYMENT TYPE ANALYSIS:")
    payment_laundering = df.groupby('Payment_type')['Is_laundering'].agg(['count', 'sum', 'mean'])
    payment_laundering.columns = ['Total_Txns', 'Suspicious_Txns', 'Suspicious_Rate']
    print(payment_laundering.sort_values('Suspicious_Rate', ascending=False))

    # Geographic analysis
    print(f"\n🌍 GEOGRAPHIC ANALYSIS:")
    cross_border = df['Sender_bank_location'] != df['Receiver_bank_location']
    print(f"Cross-border transactions: {cross_border.sum():,} ({cross_border.mean()*100:.1f}%)")

    # Cross-border laundering rate
    cross_border_laundering = df[cross_border]['Is_laundering'].mean()
    domestic_laundering = df[~cross_border]['Is_laundering'].mean()
    print(f"Cross-border suspicious rate: {cross_border_laundering*100:.2f}%")
    print(f"Domestic suspicious rate: {domestic_laundering*100:.2f}%")

    # Time analysis
    print(f"\n⏰ TIME ANALYSIS:")
    # Convert Date and Time
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time
    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

    # Date range
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Time span: {(df['Date'].max() - df['Date'].min()).days} days")

    # Hourly suspicious activity
    hourly_suspicious = df.groupby('Hour')['Is_laundering'].mean()
    print(f"Suspicious activity by hour:")
    print(f"Peak suspicious hour: {hourly_suspicious.idxmax()}:00 ({hourly_suspicious.max()*100:.2f}%)")
    print(f"Lowest suspicious hour: {hourly_suspicious.idxmin()}:00 ({hourly_suspicious.min()*100:.2f}%)")

    # Account analysis
    print(f"\n👤 ACCOUNT ANALYSIS:")
    unique_senders = df['Sender_account'].nunique()
    unique_receivers = df['Receiver_account'].nunique()
    print(f"Unique sender accounts: {unique_senders:,}")
    print(f"Unique receiver accounts: {unique_receivers:,}")

    # Most active accounts
    sender_activity = df['Sender_account'].value_counts()
    print(f"Most active sender: {sender_activity.iloc[0]} transactions")
    print(f"Top 1% senders account for: {sender_activity.head(int(len(sender_activity)*0.01)).sum():,} transactions")

    # Currency analysis
    print(f"\n💱 CURRENCY ANALYSIS:")
    currency_exchange = df['Payment_currency'] != df['Received_currency']
    print(f"Currency exchange transactions: {currency_exchange.sum():,} ({currency_exchange.mean()*100:.1f}%)")

    if currency_exchange.sum() > 0:
        exchange_suspicious = df[currency_exchange]['Is_laundering'].mean()
        same_currency_suspicious = df[~currency_exchange]['Is_laundering'].mean()
        print(f"Currency exchange suspicious rate: {exchange_suspicious*100:.2f}%")
        print(f"Same currency suspicious rate: {same_currency_suspicious*100:.2f}%")

    # Feature importance hints
    print(f"\n🎯 FEATURE ENGINEERING INSIGHTS:")
    print("Based on this analysis, focus on these feature types:")

    if skew(df['Amount']) > 2:
        print("✅ Amount needs log transformation (highly skewed)")

    if len(just_under) > 0:
        print("✅ Create 'just_under_threshold' feature (potential structuring)")

    if cross_border.sum() > 0:
        print("✅ Cross-border indicator is important")

    if currency_exchange.sum() > 0:
        print("✅ Currency exchange indicator is valuable")

    if hourly_suspicious.max() - hourly_suspicious.min() > 0.001:
        print("✅ Time-based features (hour, business hours) are important")

    if sender_activity.iloc[0] > sender_activity.median() * 10:
        print("✅ Account velocity features will be crucial")

    # Sample for velocity feature complexity assessment
    print(f"\n⚡ VELOCITY FEATURE COMPLEXITY:")
    sample_accounts = df['Sender_account'].value_counts().head(100).index
    sample_data = df[df['Sender_account'].isin(sample_accounts)]
    print(f"Top 100 most active accounts have {len(sample_data):,} transactions")
    print(f"This represents {len(sample_data)/len(df)*100:.1f}% of all data")
    print("💡 Velocity features will be most valuable for these high-activity accounts")

    return df

# Run the diagnostic
if __name__ == "__main__":
    # Replace with your actual file path
    df = run_diagnostic_eda("SAML-D.csv")  # or "SAML-D.csv"

🔍 Loading and analyzing AML dataset...
Dataset shape: (1670771, 12)

📊 BASIC DATASET INFO:
Total transactions: 1,670,771
Columns: ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']
Memory usage: 881.2 MB

🎯 TARGET DISTRIBUTION:
Normal transactions: 1,669,244
Suspicious transactions: 1,527
Imbalance ratio: 1:1093

❌ MISSING VALUES:
No missing values found

📋 DATA TYPES:
Time                       object
Date                       object
Sender_account              int64
Receiver_account            int64
Amount                    float64
Payment_currency           object
Received_currency          object
Sender_bank_location       object
Receiver_bank_location     object
Payment_type               object
Is_laundering               int64
Laundering_type            object
dtype: object

💰 AMOUNT ANALYSIS:
Min amount: $5.19
Max amount:

In [35]:
"""
Clean AML Data Preprocessing Pipeline
Handles full dataset efficiently with focus on most important features
"""
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import os

class AMLPreprocessor:

    def __init__(self):
        self.encoders = {}
        self.scaler = RobustScaler()
        self.feature_columns = None

        # High-risk countries based on your actual data + FATF lists
        # From your data: Albania, Nigeria, Pakistan, Morocco, Turkey show in the dataset
        self.high_risk_countries = [
            'Albania', 'Nigeria', 'Pakistan', 'Morocco', 'Turkey',  # From your data
            'Afghanistan', 'Barbados', 'Botswana', 'Burkina Faso',
            'Cambodia', 'Cayman Islands', 'Haiti', 'Iran', 'Jamaica',
            'Jordan', 'Mali', 'Myanmar', 'Nicaragua', 'Panama',
            'Philippines', 'Senegal', 'South Sudan', 'Syria',
            'Uganda', 'Yemen', 'Zimbabwe'
        ]

    def load_and_clean_data(self, csv_path):
        """
        Load and perform basic cleaning of the dataset
        """
        print(f"📖 Loading dataset from {csv_path}...")
        df = pd.read_csv(csv_path)

        print(f"Initial shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        # Basic data cleaning
        print("🧹 Cleaning data...")

        # Remove any duplicate transactions
        initial_len = len(df)
        df = df.drop_duplicates()
        if len(df) < initial_len:
            print(f"Removed {initial_len - len(df)} duplicate transactions")

        # Handle missing values
        missing_before = df.isnull().sum().sum()
        if missing_before > 0:
            print(f"Found {missing_before} missing values")
            # Fill missing categorical with 'Unknown'
            categorical_cols = df.select_dtypes(include=['object']).columns
            for col in categorical_cols:
                if col != 'Time' and col != 'Date':  # Don't fill time/date
                    df[col] = df[col].fillna('Unknown')

            # Fill missing numerical with median
            numerical_cols = df.select_dtypes(include=[np.number]).columns
            for col in numerical_cols:
                if col not in ['Is_laundering']:  # Don't fill target
                    df[col] = df[col].fillna(df[col].median())

        # Data type corrections
        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

        # Remove invalid amounts
        before_filter = len(df)
        df = df[df['Amount'] > 0]  # Remove zero or negative amounts
        df = df[df['Amount'] < 1e9]  # Remove unrealistic amounts
        after_filter = len(df)

        if before_filter != after_filter:
            print(f"Removed {before_filter - after_filter} transactions with invalid amounts")

        print(f"✅ Final cleaned dataset: {df.shape}")
        return df

    def create_features(self, df):
        """
        Create essential features for AML detection
        Focus on most important features only
        """
        print("🔧 Creating features...")

        # 1. DATETIME FEATURES
        print("  📅 Creating datetime features...")
        df['Date'] = pd.to_datetime(df['Date'])
        df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce')

        # Extract time components
        df['hour'] = df['Time'].dt.hour
        df['day_of_week'] = df['Date'].dt.dayofweek
        df['month'] = df['Date'].dt.month

        # Business logic features
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_business_hours'] = ((df['hour'] >= 9) & (df['hour'] <= 17)).astype(int)
        df['is_night_transaction'] = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)

        # 2. AMOUNT FEATURES (CRITICAL for AML)
        print("  💰 Creating amount features...")

        # Log transformation for skewed amounts
        df['amount_log'] = np.log1p(df['Amount'])

        # AML-specific amount patterns
        df['is_large_amount'] = (df['Amount'] > 10000).astype(int)  # CTR threshold
        df['is_just_under_threshold'] = ((df['Amount'] >= 9000) & (df['Amount'] < 10000)).astype(int)  # Structuring
        df['is_round_amount'] = (df['Amount'] % 1000 == 0).astype(int)  # Suspicious round amounts
        df['is_small_amount'] = (df['Amount'] < 100).astype(int)  # Micro-structuring

        # Amount percentiles for relative sizing
        df['amount_percentile'] = df['Amount'].rank(pct=True)

        # 3. GEOGRAPHIC FEATURES
        print("  🌍 Creating geographic features...")
        df['is_cross_border'] = (df['Sender_bank_location'] != df['Receiver_bank_location']).astype(int)
        df['sender_high_risk'] = df['Sender_bank_location'].isin(self.high_risk_countries).astype(int)
        df['receiver_high_risk'] = df['Receiver_bank_location'].isin(self.high_risk_countries).astype(int)
        df['involves_high_risk_country'] = ((df['sender_high_risk'] == 1) | (df['receiver_high_risk'] == 1)).astype(int)

        # 4. CURRENCY FEATURES
        print("  💱 Creating currency features...")
        df['is_currency_exchange'] = (df['Payment_currency'] != df['Received_currency']).astype(int)

        # 5. PAYMENT TYPE FEATURES (CRITICAL - Cash Deposit is 10x more suspicious!)
        print("  💳 Creating payment type features...")
        df['is_cash_transaction'] = df['Payment_type'].isin(['Cash Deposit', 'Cash Withdrawal']).astype(int)
        df['is_cash_deposit'] = (df['Payment_type'] == 'Cash Deposit').astype(int)  # Most suspicious type
        df['is_cash_withdrawal'] = (df['Payment_type'] == 'Cash Withdrawal').astype(int)
        df['is_cross_border_payment'] = (df['Payment_type'] == 'Cross-border').astype(int)
        df['is_card_payment'] = df['Payment_type'].isin(['Credit card', 'Debit card']).astype(int)
        df['is_wire_transfer'] = df['Payment_type'].isin(['ACH', 'Cross-border']).astype(int)

        # 6. SIMPLE VELOCITY FEATURES (Essential but not overly complex)
        print("  ⚡ Creating simplified velocity features...")
        df = self._create_simple_velocity_features(df)

        # 7. ENCODE CATEGORICAL VARIABLES
        print("  🏷️ Encoding categorical variables...")
        df = self._encode_categorical_features(df)

        print(f"✅ Feature engineering complete. Total features: {len(self.get_feature_columns())}")
        return df

    def _create_simple_velocity_features(self, df):
        """
        Create essential velocity features - ultra-simple approach to avoid merge issues
        """
        print("    ⚡ Computing basic velocity patterns...")

        # Sort by account and date
        df = df.sort_values(['Sender_account', 'Date', 'Time']).reset_index(drop=True)

        # Method 1: Simple transaction counting per account per day
        df['account_date_key'] = df['Sender_account'].astype(str) + '_' + df['Date'].astype(str)

        # Count transactions per account-date combination
        account_date_counts = df['account_date_key'].value_counts().to_dict()
        df['daily_txn_count'] = df['account_date_key'].map(account_date_counts)

        # Sum amounts per account-date combination
        account_date_amounts = df.groupby('account_date_key')['Amount'].sum().to_dict()
        df['daily_amount_sum'] = df['account_date_key'].map(account_date_amounts)

        # Count unique receivers per account-date
        account_date_receivers = df.groupby('account_date_key')['Receiver_account'].nunique().to_dict()
        df['unique_receivers_today'] = df['account_date_key'].map(account_date_receivers)

        # Simple frequency score (daily transactions / average for this account)
        account_avg_daily = df.groupby('Sender_account')['daily_txn_count'].transform('mean')
        df['transaction_frequency_score'] = np.where(
            account_avg_daily > 0,
            df['daily_txn_count'] / account_avg_daily,
            1.0
        )

        # Clean up temporary column
        df = df.drop(['account_date_key'], axis=1)

        # Ensure no NaN values
        velocity_cols = ['daily_txn_count', 'daily_amount_sum', 'unique_receivers_today', 'transaction_frequency_score']
        for col in velocity_cols:
            if col in df.columns:
                df[col] = df[col].fillna(0)

        print("    ✅ Velocity features created successfully")
        return df

    def _encode_categorical_features(self, df):
        """
        Efficiently encode categorical variables
        """
        categorical_cols = ['Payment_type', 'Sender_bank_location', 'Receiver_bank_location',
                           'Payment_currency', 'Received_currency']

        for col in categorical_cols:
            if col in df.columns:
                if col not in self.encoders:
                    self.encoders[col] = LabelEncoder()
                    df[f'{col}_encoded'] = self.encoders[col].fit_transform(df[col].astype(str))
                else:
                    # Handle unseen categories during prediction
                    unique_values = df[col].astype(str).unique()
                    known_values = set(self.encoders[col].classes_)

                    # Encode known values, assign -1 to unknown
                    df[f'{col}_encoded'] = df[col].astype(str).apply(
                        lambda x: self.encoders[col].transform([x])[0] if x in known_values else -1
                    )

        return df

    def get_feature_columns(self):
        """
        Return list of final feature columns for model training
        """
        if self.feature_columns is None:
            self.feature_columns = [
                # Amount features
                'Amount', 'amount_log', 'amount_percentile',
                'is_large_amount', 'is_just_under_threshold', 'is_round_amount', 'is_small_amount',

                # Time features
                'hour', 'day_of_week', 'month', 'is_weekend', 'is_business_hours', 'is_night_transaction',

                # Geographic features
                'is_cross_border', 'sender_high_risk', 'receiver_high_risk', 'involves_high_risk_country',

                # Currency features
                'is_currency_exchange',

                # Payment features
                'is_cash_transaction', 'is_cash_deposit', 'is_cash_withdrawal',
                'is_cross_border_payment', 'is_card_payment', 'is_wire_transfer',

                # Velocity features
                'daily_txn_count', 'daily_amount_sum', 'unique_receivers_today', 'transaction_frequency_score',

                # Encoded categorical
                'Payment_type_encoded', 'Sender_bank_location_encoded', 'Receiver_bank_location_encoded',
                'Payment_currency_encoded', 'Received_currency_encoded'
            ]

        return self.feature_columns

    def scale_features(self, df, fit=True):
        """
        Scale numerical features using RobustScaler (better for outliers)
        """
        feature_cols = self.get_feature_columns()
        X = df[feature_cols].fillna(0)

        if fit:
            print("🔧 Fitting scaler on training data...")
            X_scaled = self.scaler.fit_transform(X)
        else:
            print("🔧 Applying pre-fitted scaler...")
            X_scaled = self.scaler.transform(X)

        # Return as DataFrame with original column names
        return pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)

    def preprocess_full_pipeline(self, csv_path, test_size=0.2, random_state=42):
        """
        Complete preprocessing pipeline
        """
        # Load and clean data
        df = self.load_and_clean_data(csv_path)

        # Create features
        df = self.create_features(df)

        # Prepare features and target
        feature_cols = self.get_feature_columns()
        X = df[feature_cols].fillna(0)
        y = df['Is_laundering']

        print(f"\n📊 PREPROCESSING SUMMARY:")
        print(f"Total samples: {len(X):,}")
        print(f"Features: {len(feature_cols)}")
        print(f"Target distribution: {y.value_counts().to_dict()}")

        # Calculate imbalance ratio
        if y.sum() > 0:
            imbalance_ratio = (len(y) - y.sum()) / y.sum()
            print(f"⚠️  EXTREME IMBALANCE: 1:{imbalance_ratio:.0f} ratio")
            print(f"💡 Recommendation: Use SMOTE + cost-sensitive learning for training")

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )

        # Scale features
        X_train_scaled = self.scale_features(pd.DataFrame(X_train, columns=feature_cols), fit=True)
        X_test_scaled = self.scale_features(pd.DataFrame(X_test, columns=feature_cols), fit=False)

        print(f"✅ Preprocessing complete!")
        print(f"Training set: {len(X_train_scaled):,} samples")
        print(f"Test set: {len(X_test_scaled):,} samples")

        return X_train_scaled, X_test_scaled, y_train, y_test

    def save_preprocessor(self, save_path="models/"):
        """
        Save the preprocessor components
        """
        os.makedirs(save_path, exist_ok=True)

        joblib.dump(self.encoders, os.path.join(save_path, 'encoders.pkl'))
        joblib.dump(self.scaler, os.path.join(save_path, 'scaler.pkl'))
        joblib.dump(self.get_feature_columns(), os.path.join(save_path, 'feature_columns.pkl'))

        print(f"✅ Preprocessor saved to {save_path}")

    def load_preprocessor(self, load_path="models/"):
        """
        Load the preprocessor components
        """
        self.encoders = joblib.load(os.path.join(load_path, 'encoders.pkl'))
        self.scaler = joblib.load(os.path.join(load_path, 'scaler.pkl'))
        self.feature_columns = joblib.load(os.path.join(load_path, 'feature_columns.pkl'))

        print(f"✅ Preprocessor loaded from {load_path}")

# Usage example
if __name__ == "__main__":
    # Initialize preprocessor
    preprocessor = AMLPreprocessor()

    # Run full preprocessing pipeline
    X_train, X_test, y_train, y_test = preprocessor.preprocess_full_pipeline(
        csv_path="SAML-D.csv",  # Replace with your file path
        test_size=0.2,
        random_state=42
    )

    # Save preprocessor for later use
    preprocessor.save_preprocessor()

    print("\n🎯 READY FOR MODEL TRAINING!")
    print(f"Features created: {len(preprocessor.get_feature_columns())}")
    print(f"Training data shape: {X_train.shape}")
    print(f"Test data shape: {X_test.shape}")

    print(f"\n💡 KEY INSIGHTS FOR TRAINING:")
    print(f"🔥 Cash Deposits are 10x more suspicious - most important feature")
    print(f"🌍 Cross-border transactions are 3.7x more suspicious")
    print(f"💱 Currency exchange transactions are 4x more suspicious")
    print(f"⚠️  Extreme imbalance (1:1093) - needs aggressive SMOTE + cost-sensitive learning")

📖 Loading dataset from SAML-D.csv...
Initial shape: (1670771, 12)
Columns: ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']
🧹 Cleaning data...
✅ Final cleaned dataset: (1670771, 12)
🔧 Creating features...
  📅 Creating datetime features...
  💰 Creating amount features...
  🌍 Creating geographic features...
  💱 Creating currency features...
  💳 Creating payment type features...
  ⚡ Creating simplified velocity features...
    ⚡ Computing basic velocity patterns...
    ✅ Velocity features created successfully
  🏷️ Encoding categorical variables...
✅ Feature engineering complete. Total features: 33

📊 PREPROCESSING SUMMARY:
Total samples: 1,670,771
Features: 33
Target distribution: {0: 1669244, 1: 1527}
⚠️  EXTREME IMBALANCE: 1:1093 ratio
💡 Recommendation: Use SMOTE + cost-sensitive learning for training
🔧 Fitting scaler on training 

In [38]:
"""
AML Model Training Script - Optimized for Extreme Imbalance (1:1093)
Combines SMOTE + Cost-Sensitive Learning + Recall Optimization
"""
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_recall_curve
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

class AMLModelTrainer:

    def __init__(self):
        self.model = None
        self.best_threshold = 0.5

    def load_preprocessed_data(self, data_path="models/"):
        """
        Load preprocessed training data
        """
        print("📖 Loading preprocessed data...")

        # For now, we'll assume you saved the data during preprocessing
        # In a real scenario, you'd save X_train, X_test, y_train, y_test
        print("⚠️  Note: Make sure you have X_train, X_test, y_train, y_test from preprocessing")
        print("This function assumes data is passed directly to train_model()")

    def handle_extreme_imbalance(self, X_train, y_train):
        """
        Handle 1:1093 imbalance using smart SMOTE + class weights
        """
        print(f"📊 Original training distribution:")
        print(f"Normal: {(y_train == 0).sum():,}")
        print(f"Suspicious: {(y_train == 1).sum():,}")
        print(f"Ratio: 1:{(y_train == 0).sum() // (y_train == 1).sum()}")

        # Strategy: Moderate SMOTE + Heavy class weights
        print("\n⚖️  Applying smart rebalancing strategy...")

        # Don't fully balance with SMOTE (would create too much synthetic data)
        # Target ratio: 1:20 instead of 1:1093
        minority_count = (y_train == 1).sum()
        majority_count = (y_train == 0).sum()
        target_minority = majority_count // 20  # Much more manageable ratio

        if minority_count < target_minority:
            print(f"🔄 SMOTE: Increasing suspicious samples from {minority_count:,} to {target_minority:,}")

            # Use SMOTE with k_neighbors adjusted for small minority class
            k_neighbors = min(5, minority_count - 1) if minority_count > 1 else 1

            smote = SMOTE(
                sampling_strategy={1: target_minority},  # Only oversample minority class
                random_state=42,
                k_neighbors=k_neighbors
            )

            X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

            print(f"✅ After SMOTE:")
            print(f"Normal: {(y_train_balanced == 0).sum():,}")
            print(f"Suspicious: {(y_train_balanced == 1).sum():,}")
            print(f"New ratio: 1:{(y_train_balanced == 0).sum() // (y_train_balanced == 1).sum()}")

            return X_train_balanced, y_train_balanced
        else:
            print("✅ Sufficient suspicious samples, skipping SMOTE")
            return X_train, y_train

    def create_model(self, X_train_balanced, y_train_balanced):
        """
        Create XGBoost model optimized for recall
        """
        print("\n🚀 Creating XGBoost model optimized for recall...")

        # Calculate class weights for remaining imbalance
        normal_count = (y_train_balanced == 0).sum()
        suspicious_count = (y_train_balanced == 1).sum()

        if suspicious_count > 0:
            scale_pos_weight = normal_count / suspicious_count
        else:
            scale_pos_weight = 1

        print(f"📊 Class weight (scale_pos_weight): {scale_pos_weight:.2f}")

        # XGBoost optimized for imbalanced data and recall
        self.model = xgb.XGBClassifier(
            # Basic parameters
            n_estimators=200,  # Reduced for faster training
            max_depth=6,
            learning_rate=0.1,

            # Imbalance handling
            scale_pos_weight=scale_pos_weight,  # Address remaining imbalance

            # Regularization (prevent overfitting)
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=1,
            reg_lambda=1,

            # Performance
            random_state=42,
            n_jobs=-1,

            # Use simpler eval_metric for compatibility
            eval_metric='auc'
        )

        return self.model

    def train_model(self, X_train, X_test, y_train, y_test, target_recall=0.85):
        """
        Complete training pipeline with recall optimization
        """
        print("🎯 Starting AML Model Training...")
        print(f"Target recall: {target_recall*100}%")

        # Step 1: Handle imbalance
        X_train_balanced, y_train_balanced = self.handle_extreme_imbalance(X_train, y_train)

        # Step 2: Create model
        model = self.create_model(X_train_balanced, y_train_balanced)

        # Step 3: Train with early stopping
        print("\n🏋️  Training model with early stopping...")

        # Split balanced training data for early stopping
        from sklearn.model_selection import train_test_split
        X_bal_train, X_bal_val, y_bal_train, y_bal_val = train_test_split(
            X_train_balanced, y_train_balanced,
            test_size=0.15, random_state=42, stratify=y_train_balanced
        )

        # Train with early stopping (updated for newer XGBoost)
        try:
            # Try new XGBoost syntax first
            from xgboost.callback import EarlyStopping
            model.fit(
                X_bal_train, y_bal_train,
                eval_set=[(X_bal_val, y_bal_val)],
                callbacks=[EarlyStopping(rounds=20)],
                verbose=False
            )
        except:
            # Fallback to simple training without early stopping
            print("⚠️  Early stopping not available, training without it...")
            model.fit(X_bal_train, y_bal_train, verbose=False)

        # Check if model has best_iteration (only with early stopping)
        if hasattr(model, 'best_iteration') and model.best_iteration:
            print(f"✅ Training completed. Best iteration: {model.best_iteration}")
        else:
            print(f"✅ Training completed. Used all {model.n_estimators} estimators")

        # Step 4: Optimize threshold for recall
        print(f"\n🎯 Optimizing threshold for {target_recall*100}% recall...")
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        # Find optimal threshold
        precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

        # Find threshold that gives at least target recall
        valid_recall_mask = recall[:-1] >= target_recall

        if np.any(valid_recall_mask):
            # Among thresholds that achieve target recall, pick one with best precision
            best_threshold_idx = np.argmax(precision[:-1][valid_recall_mask])
            # Get the actual index in the original arrays
            valid_indices = np.where(valid_recall_mask)[0]
            actual_best_idx = valid_indices[best_threshold_idx]
            self.best_threshold = thresholds[actual_best_idx]
            achieved_recall = recall[actual_best_idx]
            achieved_precision = precision[actual_best_idx]
        else:
            # If target recall not achievable, use threshold that maximizes recall
            best_recall_idx = np.argmax(recall[:-1])
            self.best_threshold = thresholds[best_recall_idx]
            achieved_recall = recall[best_recall_idx]
            achieved_precision = precision[best_recall_idx]
            print(f"⚠️  Could not achieve {target_recall*100}% recall")

        print(f"✅ Optimal threshold: {self.best_threshold:.4f}")
        print(f"📊 Achieved recall: {achieved_recall:.3f} ({achieved_recall*100:.1f}%)")
        print(f"📊 Achieved precision: {achieved_precision:.3f} ({achieved_precision*100:.1f}%)")

        # Step 5: Final evaluation
        y_pred = (y_pred_proba >= self.best_threshold).astype(int)

        print(f"\n📈 FINAL MODEL PERFORMANCE:")
        print("="*50)
        print(classification_report(y_test, y_pred, target_names=['Normal', 'Suspicious']))

        print(f"\n🎯 CONFUSION MATRIX:")
        cm = confusion_matrix(y_test, y_pred)
        print(cm)
        print(f"True Negatives: {cm[0,0]:,}")
        print(f"False Positives: {cm[0,1]:,}")
        print(f"False Negatives: {cm[1,0]:,}")
        print(f"True Positives: {cm[1,1]:,}")

        # Calculate key metrics
        final_recall = recall_score(y_test, y_pred)
        suspicious_caught = cm[1,1]
        suspicious_total = cm[1,0] + cm[1,1]

        print(f"\n🏆 KEY RESULTS:")
        print(f"🎯 Recall (Suspicious Detection): {final_recall:.3f} ({final_recall*100:.1f}%)")
        print(f"🔍 Suspicious Transactions Caught: {suspicious_caught}/{suspicious_total}")
        print(f"📊 Optimal Threshold: {self.best_threshold:.4f}")

        # Feature importance
        self.show_feature_importance(model, X_train.columns)

        # Save model
        self.save_model(model)

        return model, self.best_threshold, final_recall

    def show_feature_importance(self, model, feature_names, top_n=15):
        """
        Display top feature importances
        """
        print(f"\n🔝 TOP {top_n} MOST IMPORTANT FEATURES:")
        print("="*50)

        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)

        for i, (_, row) in enumerate(feature_importance.head(top_n).iterrows()):
            print(f"{i+1:2d}. {row['feature']:<25} {row['importance']:.4f}")

        # Save feature importance
        feature_importance.to_csv('models/feature_importance.csv', index=False)
        print(f"\n💾 Feature importance saved to models/feature_importance.csv")

    def save_model(self, model):
        """
        Save trained model and threshold
        """
        print(f"\n💾 Saving model...")
        os.makedirs('models', exist_ok=True)

        # Save model
        joblib.dump(model, 'models/trained_aml_model.pkl')

        # Save threshold
        joblib.dump(self.best_threshold, 'models/optimal_threshold.pkl')

        # Save model metadata
        metadata = {
            'model_type': 'XGBoost Classifier',
            'optimal_threshold': self.best_threshold,
            'features_count': len(model.feature_names_in_) if hasattr(model, 'feature_names_in_') else 'unknown',
            'training_approach': 'SMOTE + Cost-Sensitive Learning',
            'optimization_target': 'Recall (Suspicious Detection)'
        }

        joblib.dump(metadata, 'models/model_metadata.pkl')

        print(f"✅ Model saved to models/trained_aml_model.pkl")
        print(f"✅ Threshold saved to models/optimal_threshold.pkl")
        print(f"✅ Metadata saved to models/model_metadata.pkl")

# Usage function
def train_aml_model_pipeline(X_train, X_test, y_train, y_test, target_recall=0.85):
    """
    Main training pipeline function
    """
    trainer = AMLModelTrainer()
    model, threshold, final_recall = trainer.train_model(
        X_train, X_test, y_train, y_test, target_recall
    )

    if final_recall >= 0.80:
        print(f"\n🎉 EXCELLENT! Achieved {final_recall*100:.1f}% recall!")
        print("🚀 Your model is ready for production!")
    elif final_recall >= 0.70:
        print(f"\n👍 GOOD! Achieved {final_recall*100:.1f}% recall!")
        print("💡 Consider collecting more suspicious transaction examples for improvement")
    else:
        print(f"\n⚠️  MODERATE: {final_recall*100:.1f}% recall achieved")
        print("🔧 Consider adjusting class weights or collecting more data")

    return model, threshold, final_recall

# Example usage (you'll call this with your preprocessed data)
if __name__ == "__main__":
    print("🎯 AML Model Training Script")
    print("Call train_aml_model_pipeline(X_train, X_test, y_train, y_test)")
    print("with your preprocessed data from the previous step")

🎯 AML Model Training Script
Call train_aml_model_pipeline(X_train, X_test, y_train, y_test)
with your preprocessed data from the previous step


In [39]:
model, threshold, final_recall = train_aml_model_pipeline(X_train, X_test, y_train, y_test)

🎯 Starting AML Model Training...
Target recall: 85.0%
📊 Original training distribution:
Normal: 1,335,394
Suspicious: 1,222
Ratio: 1:1092

⚖️  Applying smart rebalancing strategy...
🔄 SMOTE: Increasing suspicious samples from 1,222 to 66,769
✅ After SMOTE:
Normal: 1,335,394
Suspicious: 66,769
New ratio: 1:20

🚀 Creating XGBoost model optimized for recall...
📊 Class weight (scale_pos_weight): 20.00

🏋️  Training model with early stopping...
⚠️  Early stopping not available, training without it...
✅ Training completed. Used all 200 estimators

🎯 Optimizing threshold for 85.0% recall...
✅ Optimal threshold: 0.0446
📊 Achieved recall: 0.852 (85.2%)
📊 Achieved precision: 0.006 (0.6%)

📈 FINAL MODEL PERFORMANCE:
              precision    recall  f1-score   support

      Normal       1.00      0.86      0.93    333850
  Suspicious       0.01      0.85      0.01       305

    accuracy                           0.86    334155
   macro avg       0.50      0.86      0.47    334155
weighted avg 